# RAGAS reference-free context metric -- Dual-Level Retrieval + Hybrid RAG (interactive)

Companion to `test_ragas_metrics.ipynb` (PPR + Hybrid) and `test_ragas_metrics_hybrid_only.ipynb` (Hybrid only) -- same metric, same evaluator setup, but scores the **Dual-Level Retrieval (LightRAG local+global) + Hybrid RAG, fused** pipeline (`run_dual_level_hybrid_question.py`) instead, run against a different question set: `data/eval_singlehop_lightrag_v2.jsonl` (100 questions, LightRAG-style `local`/`global` mode questions -- entity-overview and thematic/aggregation questions, not the multi-hop set the other two notebooks use).

Deliberately bypasses this pipeline's usual Azure OpenAI / APIM setup -- evaluates via plain OpenAI (`OPENAI_API_KEY` from `.env`) instead, using a separate credential from the rest of the app's LLM calls.

Uses the current (non-deprecated) `ragas==0.4.3` API (`ragas.metrics.collections.ContextPrecisionWithoutReference` + `ragas.llms.llm_factory(..., provider="openai", ...)`) -- see `test_ragas_metrics.ipynb`'s title cell for the full detail on why (deprecated-name mixup in ragas's own warning message, `provider="azure"` bug in `_get_instructor_client`, async-client requirement, etc.) if you need it again.

Only the reference-free precision metric is used, deliberately -- ragas has no "recall without reference" metric, and this dataset's `reference_answer` is being treated as unreliable here, so recall is skipped entirely.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

CHATBOT_ROOT = Path.cwd().resolve()
for _candidate in [CHATBOT_ROOT, *CHATBOT_ROOT.parents]:
    if (_candidate / "src" / "pipeline").exists():
        CHATBOT_ROOT = _candidate
        break
REPO_ROOT = CHATBOT_ROOT.parent
if str(CHATBOT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHATBOT_ROOT))
print("CHATBOT_ROOT =", CHATBOT_ROOT)

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")  # populates os.environ -- OPENAI_API_KEY lives there

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecisionWithoutReference

from run_dual_level_hybrid_question import run_dual_level_hybrid

CHATBOT_ROOT = /Users/senghok/Projects/gds_tender_test/chatbot


/opt/anaconda3/envs/gds_test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EVAL_MODEL = os.getenv("RAGAS_EVAL_MODEL", "gpt-4o-mini")

DEFAULT_QUESTION = (
    "What are all the items that must be provided to the Company or workers, "
    "and under what conditions or requirements?"
)

QUESTIONS_PATH = REPO_ROOT / "data" / "eval_singlehop_lightrag_v2.jsonl"
RESULTS_PATH = CHATBOT_ROOT / "domain_data" / "eval_results" / "ragas_context_precision_singlehop_lightrag_v2_dual_level_hybrid.json"

In [3]:
def get_evaluator_llm():
    """Plain OpenAI (not this app's Azure/APIM setup), authenticated via
    OPENAI_API_KEY. Must be an async client -- ascore() calls
    self.llm.agenerate() internally, which raises "Cannot use agenerate()
    with a synchronous client" against a sync client."""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY not set (checked process env and .env at repo root)")
    async_client = AsyncOpenAI(api_key=api_key)
    return llm_factory(EVAL_MODEL, provider="openai", client=async_client)


evaluator_llm = get_evaluator_llm()
print(f"evaluator_llm ready (model={EVAL_MODEL!r})")

evaluator_llm ready (model='gpt-4o-mini')


## Toy example (Eiffel Tower)

The example from the ragas docs -- validates the setup works at all before trusting it against real pipeline output.

In [ ]:
async def test_toy_example(evaluator_llm) -> None:
    print("=" * 100)
    print("Toy example (Eiffel Tower)")
    print("=" * 100)

    metric = ContextPrecisionWithoutReference(llm=evaluator_llm)
    result = await metric.ascore(
        user_input="Where is the Eiffel Tower located?",
        response="The Eiffel Tower is located in Paris.",
        retrieved_contexts=["The Eiffel Tower is located in Paris."],
    )
    print(f"ContextPrecisionWithoutReference: {result.value}")
    return result

In [ ]:
toy_result = await test_toy_example(evaluator_llm)

## Real pipeline question (Dual-Level Retrieval + Hybrid RAG, fused)

Runs one question through `run_dual_level_hybrid_question.py` (LightRAG local+global retrieval fused with Hybrid RAG via RRF) and scores its actual retrieved contexts + answer. Change `question` in the run cell and re-run to try others without rebuilding `evaluator_llm`.

In [ ]:
async def test_real_question(evaluator_llm, question: str):
    print("=" * 100)
    print(f"Real pipeline question: {question}")
    print("=" * 100)

    result = run_dual_level_hybrid(question)
    answer = result["answer"]
    contexts = [c.get("content", "") for c in result["reranked_chunks"] if c.get("content")]
    print(f"Retrieved contexts: {len(contexts)}")
    print(f"Answer: {answer.answer[:300]}")
    print()

    metric = ContextPrecisionWithoutReference(llm=evaluator_llm)
    score = await metric.ascore(
        user_input=question, response=answer.answer, retrieved_contexts=contexts,
    )
    print(f"ContextPrecisionWithoutReference: {score.value}")
    return {"answer": answer, "contexts": contexts, "score": score}

In [ ]:
question = DEFAULT_QUESTION
real_result = await test_real_question(evaluator_llm, question)

## Batch eval: all questions in `data/eval_singlehop_lightrag_v2.jsonl`

Runs every question through the Dual-Level + Hybrid pipeline and scores each with `ContextPrecisionWithoutReference`, sequentially (`rerank_chunks`' cross-encoder isn't thread-safe on this machine -- same constraint as every other pipeline eval script in `scripts/domain_eval/`; note `run_dual_level_hybrid_question.py` currently has its `rerank_chunks` call commented out and passes `fused` straight through, so this constraint is moot for this particular runner but kept sequential for consistency with the others). Set `LIMIT` below to a small number first to sanity-check before the full run.

In [4]:
def load_eval_questions(limit=None):
    rows = []
    with QUESTIONS_PATH.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows[:limit] if limit else rows


LIMIT = None  # set e.g. 5 for a quick smoke test before the full 100-question run
eval_questions = load_eval_questions(limit=LIMIT)
print(f"{len(eval_questions)} questions loaded from {QUESTIONS_PATH}")

100 questions loaded from /Users/senghok/Projects/gds_tender_test/data/eval_singlehop_lightrag_v2.jsonl


In [6]:
async def run_batch_ragas_eval(evaluator_llm, questions):
    """Sequential -- run_dual_level_hybrid uses the same fusion/reranking
    machinery as the other pipeline eval scripts, so questions are scored
    one at a time, not concurrently (same constraint as every other pipeline
    eval script in scripts/domain_eval/). A single question's failure is
    caught and logged, not fatal to the whole batch."""
    metric = ContextPrecisionWithoutReference(llm=evaluator_llm)
    results = []
    for i, item in enumerate(questions, 1):
        question = item["question"]
        print(f"[{i}/{len(questions)}] {question[:80]}")
        try:
            result = run_dual_level_hybrid(question)
            answer = result["answer"]
            contexts = [c.get("content", "") for c in result["reranked_chunks"] if c.get("content")]
            score = await metric.ascore(
                user_input=question, response=answer.answer, retrieved_contexts=contexts,
            )
            results.append({
                "id": i,
                "question": question,
                "reference_answer": item.get("reference_answer"),
                "question_type": item.get("question_type"),
                "mode": item.get("mode"),
                "ok": True,
                "answer": answer.answer,
                "n_contexts": len(contexts),
                "context_precision_without_reference": score.value,
            })
            print(f"    context_precision_without_reference={score.value:.4f}")
        except Exception as exc:
            print(f"    FAILED: {type(exc).__name__}: {exc}")
            results.append({
                "id": i, "question": question, "ok": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
    return results

In [ ]:
batch_results = await run_batch_ragas_eval(evaluator_llm, eval_questions)

[1/100] What are all the items that must be provided to the Company or workers, and unde


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n            CALL db.index.vector.queryNodes('relation_embedding_index', $query_k, $qvec)\n            YIELD node AS r, score\n            WHERE ($domains IS NULL OR size([(r)-[:CLASSIFIED_AS]->(dom) WHERE dom.domainName IN $domains | dom]) > 0)\n            MATCH (s:Entity)-[:SUBJECT_OF]->(r)-[:OBJECT_OF]->(o:Entity)\n            OPTIO

    context_precision_without_reference=1.0000
[2/100] What are Lump Sum prices, and how are they included in the initial Contract Pric


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n            CALL db.index.vector.queryNodes('relation_embedding_index', $query_k, $qvec)\n            YIELD node AS r, score\n            WHERE ($domains IS NULL OR size([(r)-[:CLASSIFIED_AS]->(dom) WHERE dom.domainName IN $domains | dom]) > 0)\n            MATCH (s:Entity)-[:SUBJECT_OF]->(r)-[:OBJECT_OF]->(o:Entity)\n            OPTIO

    context_precision_without_reference=0.9743
[3/100] What are all the knowledge and training requirements for Contractor HSES personn


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n            CALL db.index.vector.queryNodes('relation_embedding_index', $query_k, $qvec)\n            YIELD node AS r, score\n            WHERE ($domains IS NULL OR size([(r)-[:CLASSIFIED_AS]->(dom) WHERE dom.domainName IN $domains | dom]) > 0)\n            MATCH (s:Entity)-[:SUBJECT_OF]->(r)-[:OBJECT_OF]->(o:Entity)\n            OPTIO

    context_precision_without_reference=1.0000
[5/100] What are the mandatory inclusions specified across key project plans and program


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n            CALL db.index.vector.queryNodes('relation_embedding_index', $query_k, $qvec)\n            YIELD node AS r, score\n            WHERE ($domains IS NULL OR size([(r)-[:CLASSIFIED_AS]->(dom) WHERE dom.domainName IN $domains | dom]) > 0)\n            MATCH (s:Entity)-[:SUBJECT_OF]->(r)-[:OBJECT_OF]->(o:Entity)\n            OPTIO

    context_precision_without_reference=1.0000
[6/100] What are all the regulatory approvals and plans required under specific legislat


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n            CALL db.index.vector.queryNodes('relation_embedding_index', $query_k, $qvec)\n            YIELD node AS r, score\n            WHERE ($domains IS NULL OR size([(r)-[:CLASSIFIED_AS]->(dom) WHERE dom.domainName IN $domains | dom]) > 0)\n            MATCH (s:Entity)-[:SUBJECT_OF]->(r)-[:OBJECT_OF]->(o:Entity)\n            OPTIO

In [ ]:
ok_results = [r for r in batch_results if r.get("ok")]
scores = [r["context_precision_without_reference"] for r in ok_results]
avg_score = sum(scores) / len(scores) if scores else None

summary = {
    "n_questions": len(batch_results),
    "n_ok": len(ok_results),
    "n_errors": len(batch_results) - len(ok_results),
    "avg_context_precision_without_reference": round(avg_score, 4) if avg_score is not None else None,
}

print("=" * 100)
print("BATCH RAGAS EVAL (DUAL-LEVEL + HYBRID) -- SUMMARY")
print("=" * 100)
for k, v in summary.items():
    print(f"{k:<45}{v}")

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(
    json.dumps({"summary": summary, "results": batch_results}, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"\nWrote full detail -> {RESULTS_PATH}")